In [5]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [ ]:
# Load data
df = pd.read_csv("../dataset/processed/resale_flat_price_features.csv")

# Drop leakages
drop_cols = [
    "resale_price",
    "log_resale_price",
    "price_per_sqm",
    "block",
    "street_name",
    "month"
]

X = df.drop(columns=drop_cols)

# Preprocessing numerical and categorical features
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

# Requires imputing as linear regression cannot handle NaN valuees
num_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),  # handle NaN
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),  # fill NaN
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, num_cols),
        ("cat", cat_pipeline, cat_cols)
    ]
)

# Model pipeline
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

# Train test split
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

# Evaluation metrics
def evaluate_model(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)

    return {
        "RMSE": rmse,
        "MAE": mae,
        "MAPE (%)": mape,
        "R2": r2
    }

# resale_price as target
y_price = df["resale_price"]

y_train_p, y_test_p = train_test_split(y_price, test_size=0.2, random_state=42)

model.fit(X_train, y_train_p)
y_pred_p = model.predict(X_test)

metrics_price = evaluate_model(y_test_p, y_pred_p)

# log_resale_price as target
y_log = df["log_resale_price"]

y_train_l, y_test_l = train_test_split(y_log, test_size=0.2, random_state=42)

model.fit(X_train, y_train_l)
y_pred_l = model.predict(X_test)

# convert back to price scale
y_test_l_exp = np.exp(y_test_l)
y_pred_l_exp = np.exp(y_pred_l)

metrics_log = evaluate_model(y_test_l_exp, y_pred_l_exp)

# Results
print("===== Linear Regression Results =====\n")

print("Using resale_price:")
for k, v in metrics_price.items():
    print(f"{k}: {v:.4f}")

print("\n Using log_resale_price (converted back):")
for k, v in metrics_log.items():
    print(f"{k}: {v:.4f}")

/tmp/ipykernel_1463/936319150.py:18: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = X.select_dtypes(include=["object"]).columns.tolist()


===== Linear Regression Results =====

Using resale_price:
RMSE: 44544.9030
MAE: 32432.4243
MAPE (%): 6.6065
R2: 0.9442

 Using log_resale_price (converted back):
RMSE: 39095.8121
MAE: 28030.8467
MAPE (%): 5.4774
R2: 0.9570


## Interpretation of Results

Two versions of the Linear Regression baseline model were evaluated:

1. Using **`resale_price`** as the target  
2. Using **`log_resale_price`** as the target, with predictions converted back to the original price scale for fair comparison

### Model Performance Summary

| Target Variable | RMSE | MAE | MAPE (%) | R² |
|---|---:|---:|---:|---:|
| `resale_price` | 44,544.90 | 32,432.42 | 6.61 | 0.9442 |
| `log_resale_price` (converted back) | 39,095.81 | 28,030.85 | 5.48 | 0.9570 |

### Interpretation

The model trained using **`log_resale_price`** performed better than the model trained using **`resale_price`** across all evaluation metrics.

- **RMSE** decreased from **44,544.90** to **39,095.81**, showing that the log-transformed model produced smaller large prediction errors overall.
- **MAE** decreased from **32,432.42** to **28,030.85**, which means the average absolute prediction error was reduced by about **4,400**.
- **MAPE** improved from **6.61%** to **5.48%**, indicating that the log-transformed model achieved better percentage-based prediction accuracy.
- **R²** increased from **0.9442** to **0.9570**, meaning the log-transformed model explained more variation in resale prices.

### Why `log_resale_price` Performed Better

Applying a log transformation helps to:

- reduce skewness in the target variable,
- stabilise variance,
- make the relationship between predictors and target more linear,
- reduce the influence of extreme high-price observations.